### 簡単な例

簡単な関数に対して適用します。


In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
#import matplotlib.mlab as mlab
%matplotlib inline
pd.set_option("display.max_columns", 10)
pd.set_option("display.max_rows", 10)


In [ ]:
g_n_sample = 5


以下の関数を用いた観測量をパラメタの誤差を含めてを回帰します。
$$
T = -0.3 X_1 + 0.1 \sin(10X_2) + \delta
$$
つまり基底関数は$(X_1, \sin(10X_2))$です。
$T$は
ガウスノイズ$\delta$を含めた観測量です。

具体的に関数を表示させます。

In [ ]:
def make_data(Nall=30, sigma=0.05, seed=11):
    """make data

    Args:
        Nall (int, optional): the number of data. Defaults to 30.
        sigma (float, optional): noise. Defaults to 0.05.
        seed (int, optional): random seed. Defaults to 11.

    Returns:
        pd.DataFrame: data X1, X2, Y0, T, err, Y0 is true target values
    """
    np.random.seed(11)

    Xrange = [-1, 1]
    # noise ,  std dev of gaussian

    w = [-0.3, 0.1]

    X1 = np.linspace(Xrange[0], Xrange[1], Nall)
    X2 = np.sin(10*X1)
    Xall = np.vstack([X1, X2]).T
    Y0 = np.dot(Xall, w)
    Tall = np.random.normal(Y0, scale=sigma, size=Nall)

    df = pd.DataFrame({"x1": X1, "x2": X2, "y": Y0, "T": Tall, "yerr": sigma})

    return df


g_sigma = 0.05
g_df = make_data(sigma=g_sigma)
g_df


In [ ]:
g_X = g_df[["x1", "x2"]].values
g_y = g_df["y"].values
g_T = g_df["T"].values
g_yerr = g_df["yerr"].values


可視化します。

In [ ]:
# 表示
def show_Xyyerr(X, y, T, yerr):
    """show X, y and T and yerr.
    
    Args:
        X (np.ndarray): X.
        y (np.ndarray): y.
        T (np.ndarray): T.
        yerr (np.ndarray): observation error of y.
    """
    fig, ax = plt.subplots()
    ax.errorbar(X[:, 0], y, yerr=yerr, fmt=".-",
                capsize=5, label="Y0,bar=std dev.")
    ax.plot(X[:, 0], T, ".-", label="T")
    ax.legend()


show_Xyyerr(g_X, g_y, g_T, g_yerr)


回帰するためにランダムに重複の無い５点とります。

In [ ]:
def get_sample(nall, n_sample=5, seed=11):
    """get samples.

    Args:
        nall (int): number of all observations.
        n_sample (int, optional): the number of samples. Defaults to 5.
        seed (int, optional): initial random seed. Defaults to 11.

    Returns:
        [int]: sample indeces.
    """
    idx = list(range(nall))
    np.random.seed(seed)

    idx_shuffle = np.random.permutation(idx)
    idx_select = idx_shuffle[:n_sample]
    return idx_select


g_idx_select = get_sample(g_X.shape[0], n_sample=g_n_sample)
#idx_select = np.random.choice(idx, n_sample)
g_X_select = g_X[g_idx_select]
g_T_select = g_T[g_idx_select]


In [ ]:
# 規格化関数の定義
def normalize_X(X):
    from sklearn.preprocessing import StandardScaler 
    scaler = StandardScaler()
    scaler.fit(X)
    X2 = scaler.transform(X)
    return X2


線形回帰での係数を見ておきます。

In [ ]:
from sklearn.linear_model import LinearRegression

def fit_linearRegression(X, y):
    """fit data with linear regression

    Args:
        X (np.data): descriptor
        y (np.data): target values

    Returns:
        np.array: coefficients of the linear model
    """
    X = normalize_X(X.copy()) # 関数内部でだけ規格化する。
    reg = LinearRegression(fit_intercept=False,)
    reg.fit(X, y)
    print("coef=", reg.coef_.ravel(), "R2=", reg.score(X, y))
    return reg.coef_.ravel()


g_linear_coef = fit_linearRegression(g_X_select, g_T_select)


この解を可視化しておきます。


In [ ]:
def predict_and_plot(X, T, Xall, Y0, linear_coef):
    """prediction and plot it

    Args:
        X (np.array): training descriptor
        T (np.array): experimental target values with noise
        Xall (np.array): all the descriptor
        Y0 (np.array): true target values
        linear_coef (np.array): the coefficients of the linear model
    """
    fig, ax = plt.subplots()
    ypall = np.dot(Xall, linear_coef)
    ax.plot(Xall[:, 0], ypall, "-", label="pred.")
    ax.plot(X[:, 0], T, "o", label="selected", color="blue")
    ax.plot(Xall[:, 0], Y0, "-", label="Y0")
    ax.legend()


predict_and_plot(g_X_select, g_T_select, g_X, g_y, g_linear_coef)


通常はこの回帰はありえないでしょう。しかし、線形回帰モデルの制限が強いので、このような回帰モデルが求ます。

#### データ解析

以下では回帰係数の分布を調べるためにベイス線形回帰を行います。

$w$がsize2 vectorなので、$S_N$は2x2行列,$m_N$はsize 2 vectorとなります

まず、一度に解く手法を用います。

In [ ]:
def solve_once(X, T, sigma_, w0, sigma0):
    """一度に解く
    式(2)(3)

    Args:
        X (np.array): descriptor
        T (np.array): observed target values
        sigma_ (float): a value of sigma for S
        w0 (np.array): inital coefficients
        sigma0 (float): a value of sigma for beta

    Returns:
        list: S
        list: m
    """
    m_0 = w0.copy()
    S_0 = np.identity(w0.shape[0]) / sigma0**2

    beta_inv = np.identity(w0.shape[0])/sigma_**2
    print("n,bar_S_N, bar_m_N", 0, S_0, m_0)

    Slist = []
    mlist = []
    Slist.append(S_0.copy())
    mlist.append(m_0.copy())

    w2coef = np.linalg.inv(S_0)
    w1coef = m_0.copy()

    for i, (x_N, t_N) in enumerate(zip(X, T)):
        # print(x_N,t_N,beta_inv)
        w2coef += x_N.reshape(-1, 1) * np.dot(beta_inv, x_N)
        w1coef += np.dot(x_N,    beta_inv) * t_N

        S = np.linalg.inv(w2coef)
        m = np.dot(S, w1coef)

        Slist.append(S.copy())
        mlist.append(m.copy())
    print(i, ",bar_S_N, bar_m_N", i, S, m)
    return Slist, mlist


g_w0 = np.array([0.0, 0.0])
g_sigma0 = 5.0

g_Slist1, g_mlist1 = solve_once(
    g_X_select, g_T_select, g_sigma, g_w0, g_sigma0)


逐次解法を用いて解を求めます。

In [ ]:
from scipy.stats import multivariate_normal


def solve_iteratively(X, T, sigma_, w0, sigma0,):
    """逐次計算手法を用いる。
    式(6),(7)

    Args:
        X (np.array): descriptor
        T (np.array): observed target values
        sigma_ (float): a value of sigma for S
        w0 (np.array): inital coefficients
        sigma0 (float): a value of sigma for beta

    Returns:
        list: S
        list: m
    """

    # 初期状態 平均(0,0),stddev = (0.1,0.1)
    m_0 = w0.copy()
    S_0 = np.identity(w0.shape[0]) / sigma0**2

    beta_inv = np.identity(w0.shape[0])/sigma_**2

    # save data
    Slist = []
    mlist = []

    bar_m_N = m_0.copy()
    bar_S_N = S_0.copy()
    print("n,bar_S_N,bar_m_N=", 0, bar_S_N, bar_m_N)
    Slist.append(bar_S_N)
    mlist.append(bar_m_N)

    # fit

    for i, (x_N, t_N) in enumerate(zip(X, T)):

        bar_m_N1 = bar_m_N.copy()
        bar_S_N1 = bar_S_N.copy()

        # estimate parameters
        bar_S_N1_inv = np.linalg.inv(bar_S_N1)
        bar_S_N_inv = bar_S_N1_inv + x_N.reshape(-1, 1) * np.dot(beta_inv, x_N)

        bar_S_N = np.linalg.inv(bar_S_N_inv)

        bar_m_N = np.dot(bar_S_N,
                         np.dot(bar_S_N1_inv, bar_m_N1) + np.dot(x_N, np.dot(beta_inv, t_N)))
        print(i, ",bar_S_N,bar_m_N= ")
        print(bar_S_N, bar_m_N)

        Slist.append(bar_S_N)
        mlist.append(bar_m_N)

    return Slist, mlist


g_Slist2, g_mlist2 = solve_iteratively(
    g_X_select, g_T_select, g_sigma, g_w0, g_sigma0)


先ほどの係数の解と比較します。同じ解であることがわかります。

In [ ]:
g_Slist1[-1], g_mlist1[-1]


#### 可視化

multivariate_normalを用いてn個のランダムな係数から回帰曲線を書いてみます。

In [ ]:
from numpy.random import multivariate_normal


def show_curves(bar_S_N, bar_m_N, Xall, Yall, X, T):
    """show predicted curves

    Args:
        bar_S_N (list): bar S_N
        bar_m_N (list): bar m_N
        Xall (np.array): all the X points
        Yall (np.array): true y points
        X (np.array): X points for T
        T (np.array): observed points for X
    """
    fig, ax = plt.subplots()

    n = 200
    # mean, covariantの分布からrandomにn個引く。
    w_rand = multivariate_normal(bar_m_N, bar_S_N, size=n)
    for i, w_rand1 in enumerate(w_rand):
        # print("w",w_rand1)
        Yall_rand = np.dot(Xall, w_rand1)
        ax.plot(Xall[:, 0], Yall_rand, "-",
                linewidth=1, color="red", alpha=0.08)

    ax.plot(Xall[:, 0], Yall, color="blue", label="Y0", linewidth=1)
    ax.plot(X[:, 0], T, "o", color="blue", label="selected")
    ax.legend()


In [ ]:
g_bar_S_N = g_Slist1[-1]
g_bar_m_N = g_mlist1[-1]
print(g_bar_S_N, g_bar_m_N)

show_curves(g_bar_S_N, g_bar_m_N, g_X, g_y, g_X_select, g_T_select)


係数の平均値だけでなく共分散まで考えると直感と一致する予測を行っていることがわかります。
しかし、全く異なる回帰モデルもありそうです。
モデルを決めたベイズ回帰や回帰で係数を決める場合は、
モデルを決めるということの条件がかなり大きいことを理解してください。

なお、priorを変えると異なる解になります。つまり、ベイズ回帰の解はpriorに依存します。

次に、漸化式に従ってTにサンプル点を加えていくと$P(w|t)$が移動していく様子を表示します。
2*std（〜95%信頼区間）を書いています。
サンプル点を加えると、広がった初期値から色が濃い収束値に移動していきます。

In [ ]:
def calculate_ellipse_param(S):
    """分散行列から長軸短軸を計算する

    Args:
        S (np.array): 分散行列

    Returns:
        float: 長軸長さ
        float: 短軸長さ
        float: 長軸の角度
    """
    eig, w = np.linalg.eigh(S)
    i = eig.argsort()[::-1]
    eig = eig[i]
    w = w[:, i]
    angle = np.degrees(np.arctan2(*w[:, 0][::-1]))
    return 2*np.sqrt(eig[0]), 2*np.sqrt(eig[1]), angle


In [ ]:
from numpy.random import multivariate_normal
import matplotlib.patches as pat


def show_Sm(Slist2, mlist2, n_sample):
    """show S, covariance matrix, and m, mean value.

    Args:
        Slist2 ([[float]]): a list of S.
        mlist2 ([float]): a list of m.
        n_sample (int): the number of samples.
    """
    fig, ax = plt.subplots()
    for i in range(n_sample):
        print(i, "S=", Slist2[i], "m=", mlist2[i])
        m = mlist2[i]
        S = Slist2[i]
        if False:
            w_rand = multivariate_normal(m, S, size=1000)
            ax.plot(w_rand[:, 0], w_rand[:, 1], ".")
        fac_std = 2  # about 95% confidence
        width, height, angle = calculate_ellipse_param(S)
        ellipse = pat.Ellipse(m, angle=angle, width=fac_std*width,
                              height=fac_std*height, alpha=0.1)
        ax.add_patch(ellipse)
        ax.set_xlim((-0.5, 0.5))
        ax.set_ylim((-0.5, 0.5))
        ax.set_title(str(i))
    plt.show()


show_Sm(g_Slist2, g_mlist2, g_n_sample)


## 問題

n_sampleを10に変える。
